In [1]:
!pip install -U llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 23.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl size=20848345 sha256=88c6f4808b5bf08705a5216e7970b436bfb189f26086f14c3bdc1e6f59cfbbad
  Stored in directory: /root/.cache/pip/wheels/1b/64/d4/17744d793e69b485a7664ef47b18e402a72a6e08e84f7b9926
Successfully built llama-cpp-python


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv


In [3]:

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="EngineerWanga0791709020/SME-Ledger",
	filename="sme-ledger-v2-Q4_K_M.gguf",
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./sme-ledger-v2-Q4_K_M.gguf:   0%|          | 0.00/261M [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 33 key-value pairs and 236 tensors from /root/.cache/huggingface/hub/models--EngineerWanga0791709020--SME-Ledger/snapshots/1aa4d6566c59f07727d7a814269122c8f037dd09/./sme-ledger-v2-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                               general.name str              = Merged
llama_model_loader: - kv   5:                         general.size_label str              = 268M
llama_model_loader: - kv   6:                       

In [4]:
# ============================================================
# SME-LEDGER V2 — MANUAL 20-SAMPLE EVALUATION
#
# You manually judge:
#   - JSON validity
#   - Correctness
#   - Missing information handling
#   - Capability answers
#
# This script ONLY shows:
#   1. Prompt given to model
#   2. Model response
#   3. Latency
#
# No automatic JSON scoring.
# ============================================================

import json
import time
from datetime import datetime


# ============================================================
# 20 TEST CASES
# ============================================================

TESTS = [

    # ========================================================
    # CATEGORY 1 — VALID TRANSACTIONS (8)
    # ========================================================

    {
        "id": "valid_01",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract the financial transaction below.

Return ONLY valid JSON with exactly these fields:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Rules:
- amount, balance and fee must be numbers.
- type must be income, expense, or unknown.
- date must be YYYY-MM-DD.
- Do not invent missing information.
- Use null when information is unavailable.
- No markdown.
- No explanation.

SMS:
QGH7K3M2P1 Confirmed. You have received Ksh20,000.00 from Ann Mueni 0712***456 on 05/03/2026 at 10:42 AM. New M-PESA balance is Ksh159,583.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_02",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this M-Pesa payment.

Return ONLY JSON with:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Do not invent missing information. Use null where necessary.

SMS:
TLA82K9P4Q Confirmed. Ksh1,250.00 paid to Naivas Supermarket on 06/03/2026 at 14:21. New M-PESA balance is Ksh158,333.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_03",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this M-Pesa Till transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
RTP93LMN72 Confirmed. Ksh3,500.00 paid to 123456 - Wanga Electronics via M-PESA Till Number on 07/03/2026 at 09:15 AM. New M-PESA balance is Ksh154,833.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_04",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this PayBill transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
PBY72LK91A Confirmed. Ksh5,000.00 sent to KPLC via PayBill 88888 for account 123456789 on 08/03/2026 at 18:03. New M-PESA balance is Ksh149,833.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_05",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this bank transfer received.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
BNK72PQ91 Confirmed. Ksh45,000.00 received in your M-PESA account from Equity Bank on 09/03/2026 at 11:30 AM. New M-PESA balance is Ksh194,833.00. Reference EQT98431.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_06",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this Fuliza transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
FUL123ABC Confirmed. Fuliza loan of Ksh10,000.00 received on 10/03/2026 at 08:05 AM. New M-PESA balance is Ksh204,833.00. Fuliza outstanding balance is Ksh10,000.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_07",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this cash withdrawal.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
WD91KLM22 Confirmed. Ksh8,000.00 withdrawn from M-PESA at Agent 456789 - John Kamau on 11/03/2026 at 16:40. Transaction cost, Ksh80.00. New M-PESA balance is Ksh196,753.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_08",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this reversal.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
REV8821 Confirmed. Reversal of Ksh2,500.00 for transaction QWE12345 has been credited to your M-PESA account on 12/03/2026 at 13:22. New M-PESA balance is Ksh199,253.00.

Return ONLY JSON.
"""
    },


    # ========================================================
    # CATEGORY 2 — MISSING / AMBIGUOUS DATA (6)
    # ========================================================

    {
        "id": "missing_01",
        "category": "missing_data",
        "task": "Handle missing balance",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null for information that is not present.
Do not guess.

SMS:
ABC12345 Confirmed. You have received Ksh7,500.00 from Mary Wanjiku on 13/03/2026 at 09:10 AM.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_02",
        "category": "missing_data",
        "task": "Handle missing transaction ID",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null for missing information.

SMS:
You paid Ksh2,000.00 to Green Valley Shop on 14/03/2026 at 15:20. New M-PESA balance is Ksh197,253.00.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_03",
        "category": "missing_data",
        "task": "Handle missing entity",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Do not invent an entity.

SMS:
TX99821 Confirmed. Ksh3,200.00 paid via M-PESA on 15/03/2026 at 12:00 PM. New M-PESA balance is Ksh194,053.00.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_04",
        "category": "missing_data",
        "task": "Ambiguous transaction",
        "prompt": """
Determine what can safely be extracted from this message.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null when information is ambiguous or missing.
Do not invent facts.

SMS:
TX7712 Confirmed. Ksh5,000 sent. Balance Ksh100,000.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_05",
        "category": "missing_data",
        "task": "Noisy SMS",
        "prompt": """
Extract the transaction from this noisy SMS.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Ignore irrelevant text.
Do not invent missing information.

SMS:
M-PESA ALERT!!! Your account was updated. TX88K21 Confirmed. You received Ksh12,000 from Peter on 16/03/2026. Please do not share your PIN. New balance Ksh112,000.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_06",
        "category": "missing_data",
        "task": "Conflicting information",
        "prompt": """
Extract this transaction carefully.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

If the message contains conflicting information, preserve only what can be determined safely and use null where necessary.

SMS:
TX5566 Confirmed. Ksh4,000 paid to ABC Shop on 17/03/2026. New M-PESA balance is Ksh90,000. Later message says transaction amount was Ksh5,000.

Return ONLY JSON.
"""
    },


    # ========================================================
    # CATEGORY 3 — CAPABILITY / SELF-KNOWLEDGE (6)
    # ========================================================

    {
        "id": "capability_01",
        "category": "capability",
        "task": "Capabilities",
        "prompt": """
What types of financial transaction messages are you designed to understand?

Mention the transaction categories you can identify.

Answer concisely.
"""
    },

    {
        "id": "capability_02",
        "category": "capability",
        "task": "Supported fields",
        "prompt": """
What information can you extract from an M-Pesa or bank transaction message?

List the fields you are designed to identify.
"""
    },

    {
        "id": "capability_03",
        "category": "capability",
        "task": "Missing information",
        "prompt": """
What do you do when a financial SMS is missing important information such as the transaction ID, entity, balance, date, or amount?
"""
    },

    {
        "id": "capability_04",
        "category": "capability",
        "task": "Unsupported claims",
        "prompt": """
Can you access a user's bank account, M-Pesa account, contacts, internet, or private financial records directly?

Explain what you can and cannot access.
"""
    },

    {
        "id": "capability_05",
        "category": "capability",
        "task": "Role",
        "prompt": """
What is your role in the SME Ledger system?

Explain what happens after you extract a transaction from an SMS.
"""
    },

    {
        "id": "capability_06",
        "category": "capability",
        "task": "Transaction types",
        "prompt": """
Can you distinguish between income, expenses, transfers, withdrawals, merchant payments, PayBill payments, Fuliza transactions, reversals, and failed transactions?

If yes, briefly explain how.
"""
    }
]


# ============================================================
# RUN MODEL
# ============================================================

def run_model(prompt, max_tokens=256):

    start = time.time()

    try:

        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            seed=42,
            max_tokens=max_tokens
        )

        elapsed = time.time() - start

        text = response["choices"][0]["message"]["content"]

        return text.strip(), round(elapsed, 3), None

    except Exception as e:

        elapsed = time.time() - start

        return "", round(elapsed, 3), str(e)


# ============================================================
# RUN 20 TESTS
# ============================================================

results = []

print("\n")
print("=" * 90)
print("                 SME-LEDGER V2 — MANUAL 20-TEST EVALUATION")
print("=" * 90)


for i, test in enumerate(TESTS, start=1):

    print("\n\n")
    print("█" * 90)
    print(f"TEST {i}/20")
    print(f"ID       : {test['id']}")
    print(f"CATEGORY : {test['category']}")
    print(f"TASK     : {test['task']}")
    print("█" * 90)

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    print("\n")
    print("┌" + "─" * 88 + "┐")
    print("│ PROMPT GIVEN TO MODEL")
    print("└" + "─" * 88 + "┘")

    print(test["prompt"])

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    response, latency, error = run_model(test["prompt"])

    print("\n")
    print("┌" + "─" * 88 + "┐")
    print("│ MODEL RESPONSE")
    print("└" + "─" * 88 + "┘")

    if error:
        print("❌ ERROR:")
        print(error)
    else:
        print(response)

    print("\n")
    print(f"⏱️  Latency: {latency:.3f} seconds")

    # --------------------------------------------------------
    # Store raw result
    # --------------------------------------------------------

    results.append({
        "test_number": i,
        "id": test["id"],
        "category": test["category"],
        "task": test["task"],
        "prompt": test["prompt"],
        "response": response,
        "latency_seconds": latency,
        "error": error
    })


# ============================================================
# SAVE RAW RESULTS
# ============================================================

OUTPUT_FILE = "sme_ledger_20_manual_test_results.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        {
            "evaluation": {
                "name": "SME-Ledger V2 Manual 20-Test Evaluation",
                "model": "sme-ledger-v2-Q4_K_M.gguf",
                "timestamp": datetime.utcnow().isoformat() + "Z",
                "seed": 42
            },
            "results": results
        },
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# FINAL RUN SUMMARY
# ============================================================

successful = sum(
    r["error"] is None
    for r in results
)

average_latency = sum(
    r["latency_seconds"]
    for r in results
) / len(results)


print("\n\n")
print("=" * 90)
print("                         TEST RUN COMPLETE")
print("=" * 90)

print(f"""
Tests executed:       {len(results)}/20
Successful runs:      {successful}/20
Average latency:      {average_latency:.3f} seconds

No automatic correctness or JSON validation was performed.
You are the evaluator.
""")

print(f"Raw results saved to: {OUTPUT_FILE}")
print("=" * 90)



                 SME-LEDGER V2 — MANUAL 20-TEST EVALUATION



██████████████████████████████████████████████████████████████████████████████████████████
TEST 1/20
ID       : valid_01
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract the financial transaction below.

Return ONLY valid JSON with exactly these fields:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Rules:
- amount, balance and fee must be numbers.
- type must be income, expense, or unknown.
- date must be YYYY-MM-DD.
- Do not invent missing information.
- Use null when information is unavailable.
- No markdown.
- No explanation.

SMS:
QGH7K3M2P1 Confirmed. You have received

llama_perf_context_print:        load time =     900.52 ms
llama_perf_context_print: prompt eval time =     900.07 ms /   209 tokens (    4.31 ms per token,   232.20 tokens per second)
llama_perf_context_print:        eval time =    2224.76 ms /    93 runs   (   23.92 ms per token,    41.80 tokens per second)
llama_perf_context_print:       total time =    3236.55 ms /   302 tokens
llama_perf_context_print:    graphs reused =         91
Llama.generate: 5 prefix-match hit, remaining 136 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
transaction_id: QGH7K3M2P1
date: 2026-03-05
time: 10:42
type: Income
domain: bank_transfer_receive_money
entity: Ann Mueni
amount: 20000.0
balance: 0.0
fee: 0.0
reference: QGH7K3M2P1


⏱️  Latency: 3.242 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 2/20
ID       : valid_02
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract this M-Pesa payment.

Return ONLY JSON with:
transaction_id, date, time, type, domain, entity, 

llama_perf_context_print:        load time =     900.52 ms
llama_perf_context_print: prompt eval time =     582.98 ms /   136 tokens (    4.29 ms per token,   233.29 tokens per second)
llama_perf_context_print:        eval time =    2228.97 ms /    95 runs   (   23.46 ms per token,    42.62 tokens per second)
llama_perf_context_print:       total time =    2924.86 ms /   231 tokens
llama_perf_context_print:    graphs reused =         94
Llama.generate: 10 prefix-match hit, remaining 133 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
transaction_id: TLA82K9P4Q
date: 2026-03-06
time: 14:21
type: Expense
domain: M-PESA
entity: Naivas Supermarket
amount: 1250.0
balance: 158333.0
fee: 0.0
reference: TLA82K9P4Q


⏱️  Latency: 2.931 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 3/20
ID       : valid_03
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract this M-Pesa Till transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amo

...

llama_perf_context_print:        load time =     900.52 ms
llama_perf_context_print: prompt eval time =     128.49 ms /    28 tokens (    4.59 ms per token,   217.92 tokens per second)
llama_perf_context_print:        eval time =     846.84 ms /    36 runs   (   23.52 ms per token,    42.51 tokens per second)
llama_perf_context_print:       total time =    1020.11 ms /    64 tokens
llama_perf_context_print:    graphs reused =         35
Llama.generate: 4 prefix-match hit, remaining 42 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
I turn a M-PESA-Ledger-like assistant on-device, processing sensitive financial information and forwarding it to authorized financial intermediaries or cash-flow-oriented entities.


⏱️  Latency: 1.024 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 20/20
ID       : capability_06
CATEGORY : capability
TASK     : Transaction types
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Can you distinguish between income, expenses, transfers, withdrawals, merchant payments, PayBill payments, F

llama_perf_context_print:        load time =     900.52 ms
llama_perf_context_print: prompt eval time =     189.80 ms /    42 tokens (    4.52 ms per token,   221.28 tokens per second)
llama_perf_context_print:        eval time =    1152.73 ms /    49 runs   (   23.53 ms per token,    42.51 tokens per second)
llama_perf_context_print:       total time =    1403.23 ms /    91 tokens
llama_perf_context_print:    graphs reused =         48




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
Yes. The application can analyze financial information in a supported manner to identify transaction or balance-related patterns, such as income, expenses, cash-flow or spending imbalances. This is a preliminary step and should be tailored to the specific application structure.


⏱️  Latency: 1.409 seconds



                         TEST RUN COMPLETE

Tests executed:       20/20
Successful runs:      20/20
Average latency:      2.655 seconds

No automatic correctness or JSON validation was performed.
You are the evaluator.

Raw results saved to: sme_ledger_20_manual_test_results.json


/tmp/ipykernel_16/1229522993.py:493: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


In [5]:
# ============================================================
# SME-LEDGER V2 — PRODUCTION-STYLE 50-SMS EXTRACTION PIPELINE
#
# Input:
#   test_50_consistent.csv
#   Expected column: sms
#
# Process:
#   SMS 1 -> model -> JSON -> structured row -> append
#   SMS 2 -> model -> JSON -> structured row -> append
#   ...
#   SMS 50 -> model -> JSON -> structured row -> append
#
# Output:
#   sme_ledger_50_results.csv
#
# Uses the already-loaded `llm` from Cell 3.
# ============================================================

import pandas as pd
import json
import time
import re
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TEST_CSV = "/kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv"

OUTPUT_CSV = "/kaggle/working/sme_ledger_50_results.csv"
RAW_OUTPUT_CSV = "/kaggle/working/sme_ledger_50_raw_results.csv"

# Required output schema
JSON_FIELDS = [
    "transaction_id",
    "date",
    "time",
    "type",
    "domain",
    "entity",
    "amount",
    "balance",
    "fee",
    "reference"
]

# ------------------------------------------------------------
# 2. LOAD TEST DATA
# ------------------------------------------------------------

test_df = pd.read_csv(TEST_CSV)

print("=" * 90)
print("SME-LEDGER V2 — 50 SMS FINANCIAL LEDGER EXTRACTION")
print("=" * 90)

print(f"\nInput file: {TEST_CSV}")
print(f"Rows loaded: {len(test_df)}")
print(f"Columns: {test_df.columns.tolist()}")

# ------------------------------------------------------------
# 3. FIND SMS COLUMN
# ------------------------------------------------------------

MESSAGE_COLUMN = None

likely_names = [
    "sms",
    "message",
    "messages",
    "text",
    "transaction_message",
    "transaction",
    "raw_message"
]

normalized_columns = {
    str(c).strip().lower(): c
    for c in test_df.columns
}

for name in likely_names:
    if name in normalized_columns:
        MESSAGE_COLUMN = normalized_columns[name]
        break

if MESSAGE_COLUMN is None:
    if len(test_df.columns) == 1:
        MESSAGE_COLUMN = test_df.columns[0]
    else:
        raise ValueError(
            "Could not identify the SMS column.\n"
            f"Available columns: {test_df.columns.tolist()}\n"
            "Set MESSAGE_COLUMN manually."
        )

print(f"SMS column: {MESSAGE_COLUMN!r}")

# ------------------------------------------------------------
# 4. LIMIT TO 50 SAMPLES
# ------------------------------------------------------------

if len(test_df) > 50:
    print(f"\nDataset contains {len(test_df)} rows.")
    print("Using the first 50 samples as requested.")
    test_df = test_df.head(50).copy()

print(f"Samples to process: {len(test_df)}")

# ------------------------------------------------------------
# 5. EXTRACTION PROMPT
# ------------------------------------------------------------

def build_extraction_prompt(sms):
    return f"""
You are SME-Ledger, a financial SMS transaction extraction system.

Read the SMS carefully and extract ONLY information explicitly present
in the SMS.

Return EXACTLY ONE JSON OBJECT.

The JSON object MUST contain exactly these fields:

{{
  "transaction_id": null,
  "date": null,
  "time": null,
  "type": null,
  "domain": null,
  "entity": null,
  "amount": null,
  "balance": null,
  "fee": null,
  "reference": null
}}

Rules:

1. Return JSON only.
2. Do NOT use markdown.
3. Do NOT write explanations.
4. Do NOT add extra fields.
5. amount, balance and fee must be numbers or null.
6. type MUST be exactly one of:
   - "income"
   - "expense"
   - "unknown"
7. date MUST use YYYY-MM-DD when available.
8. time MUST use HH:MM when available.
9. If information is not explicitly present, use null.
10. Never invent names, amounts, dates, balances, references, or IDs.
11. For money values, remove "Ksh", "KES", commas, and other formatting.
12. "entity" should be the person, business, institution, agent,
    organization, or service involved in the transaction.
13. "reference" should contain a transaction/reference/account identifier
    when explicitly present.
14. "domain" should describe the transaction channel or financial domain,
    such as:
       mpesa
       bank_transfer
       paybill
       till_payment
       cash_withdrawal
       fuliza
       airtime
       unknown
15. A reversal/refund credited to the account is "income".
16. Money paid, withdrawn, or sent from the account is "expense".
17. Money received into the account is "income".
18. Loans received should still be represented as "income" because money
    entered the account; use the appropriate loan domain.

SMS:
{sms}

RETURN ONLY THE JSON OBJECT.
"""

# ------------------------------------------------------------
# 6. ROBUST JSON EXTRACTION
# ------------------------------------------------------------

def clean_model_output(text):
    """
    Clean common formatting mistakes before JSON parsing.
    """

    if text is None:
        return ""

    text = str(text).strip()

    # Remove markdown code fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    text = text.strip()

    return text


def extract_json_object(text):
    """
    Extract the first JSON object from the model response.

    Handles cases where the model adds a small amount of text around
    an otherwise valid JSON object.
    """

    cleaned = clean_model_output(text)

    # First attempt: entire response is JSON
    try:
        obj = json.loads(cleaned)

        if isinstance(obj, dict):
            return obj, None

    except Exception:
        pass

    # Second attempt: locate JSON object inside response
    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start != -1 and end != -1 and end > start:

        candidate = cleaned[start:end + 1]

        try:
            obj = json.loads(candidate)

            if isinstance(obj, dict):
                return obj, None

        except Exception as exc:
            return None, f"JSON parsing failed: {exc}"

    return None, "No valid JSON object found in model response."


# ------------------------------------------------------------
# 7. NORMALIZE JSON INTO OUR REQUIRED SCHEMA
# ------------------------------------------------------------

def normalize_transaction(data):
    """
    Force the model response into the exact SME-Ledger schema.
    """

    result = {
        field: None
        for field in JSON_FIELDS
    }

    if not isinstance(data, dict):
        return result

    # Only copy known fields
    for field in JSON_FIELDS:
        if field in data:
            result[field] = data[field]

    # Normalize type
    if result["type"] is not None:

        transaction_type = str(result["type"]).strip().lower()

        if transaction_type in ["income", "received", "receive"]:
            result["type"] = "income"

        elif transaction_type in ["expense", "paid", "payment", "withdrawal"]:
            result["type"] = "expense"

        elif transaction_type == "unknown":
            result["type"] = "unknown"

        else:
            # Unknown model label -> don't invent a category
            result["type"] = "unknown"

    # Normalize numeric fields
    for field in ["amount", "balance", "fee"]:

        value = result[field]

        if value is None:
            continue

        if isinstance(value, str):

            cleaned = (
                value
                .replace("Ksh", "")
                .replace("KES", "")
                .replace(",", "")
                .strip()
            )

            try:
                result[field] = float(cleaned)
            except Exception:
                result[field] = None

        else:

            try:
                result[field] = float(value)
            except Exception:
                result[field] = None

    # Normalize empty strings to null
    for field in JSON_FIELDS:

        if isinstance(result[field], str):

            if not result[field].strip():
                result[field] = None

    return result


# ------------------------------------------------------------
# 8. PROCESS SMS ONE BY ONE
# ------------------------------------------------------------

results = []

print("\n")
print("=" * 90)
print("STARTING SEQUENTIAL SMS EXTRACTION")
print("=" * 90)

pipeline_start = time.time()

for position, (source_index, row) in enumerate(
    test_df.iterrows(),
    start=1
):

    sms = row[MESSAGE_COLUMN]

    print("\n" + "-" * 90)
    print(f"SMS {position}/{len(test_df)}")
    print("-" * 90)

    # --------------------------------------------------------
    # Handle empty SMS
    # --------------------------------------------------------

    if pd.isna(sms) or not str(sms).strip():

        print("Status: EMPTY SMS")

        transaction = {
            field: None
            for field in JSON_FIELDS
        }

        result_row = {
            "source_row": source_index,
            "sample_number": position,
            "sms": sms,
            **transaction,
            "json_valid": False,
            "extraction_error": "Empty SMS",
            "latency_seconds": 0.0
        }

        results.append(result_row)
        continue

    sms = str(sms).strip()

    print(f"SMS preview: {sms[:180]}")

    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    prompt = build_extraction_prompt(sms)

    started = time.time()

    raw_response = ""
    transaction = {
        field: None
        for field in JSON_FIELDS
    }

    json_valid = False
    extraction_error = None

    # --------------------------------------------------------
    # MODEL INFERENCE
    # --------------------------------------------------------

    try:

        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            seed=42,

            # Keep generation controlled because we only want JSON.
            max_tokens=180
        )

        raw_response = (
            response["choices"][0]["message"]["content"]
            .strip()
        )

        # ----------------------------------------------------
        # Parse JSON
        # ----------------------------------------------------

        parsed, parse_error = extract_json_object(raw_response)

        if parsed is not None:

            transaction = normalize_transaction(parsed)

            json_valid = True

        else:

            extraction_error = parse_error

    except Exception as exc:

        extraction_error = str(exc)

    latency = round(time.time() - started, 3)

    # --------------------------------------------------------
    # BUILD FINAL STRUCTURED ROW
    # --------------------------------------------------------

    result_row = {
        "source_row": source_index,
        "sample_number": position,
        "sms": sms,

        **transaction,

        "json_valid": json_valid,
        "extraction_error": extraction_error,
        "latency_seconds": latency
    }

    results.append(result_row)

    # --------------------------------------------------------
    # SAVE AFTER EVERY SMS
    #
    # This means a failure at SMS 37 will not destroy
    # successfully processed SMS 1-36.
    # --------------------------------------------------------

    current_df = pd.DataFrame(results)

    current_df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # CONSOLE SUMMARY
    # --------------------------------------------------------

    if json_valid:

        print("Status       : ✓ JSON extracted")
        print(f"Transaction  : {transaction['transaction_id']}")
        print(f"Date         : {transaction['date']}")
        print(f"Type         : {transaction['type']}")
        print(f"Domain       : {transaction['domain']}")
        print(f"Entity       : {transaction['entity']}")
        print(f"Amount       : {transaction['amount']}")
        print(f"Balance      : {transaction['balance']}")
        print(f"Fee          : {transaction['fee']}")
        print(f"Reference    : {transaction['reference']}")

    else:

        print("Status       : ✗ EXTRACTION ERROR")
        print(f"Error        : {extraction_error}")
        print(f"Raw response : {raw_response[:300]}")

    print(f"Latency      : {latency:.3f}s")
    print(f"Saved rows   : {len(results)}")

# ------------------------------------------------------------
# 9. FINAL DATAFRAME
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

# Put columns into a clean analysis-friendly order
FINAL_COLUMNS = [
    "source_row",
    "sample_number",
    "sms",

    "transaction_id",
    "date",
    "time",
    "type",
    "domain",
    "entity",
    "amount",
    "balance",
    "fee",
    "reference",

    "json_valid",
    "extraction_error",
    "latency_seconds"
]

results_df = results_df[FINAL_COLUMNS]

# ------------------------------------------------------------
# 10. SAVE FINAL CSV
# ------------------------------------------------------------

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8"
)

# Also save a raw backup
results_df.to_csv(
    RAW_OUTPUT_CSV,
    index=False,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. SUMMARY
# ------------------------------------------------------------

total_time = time.time() - pipeline_start

successful = int(results_df["json_valid"].sum())
failed = len(results_df) - successful

print("\n")
print("=" * 90)
print("SME-LEDGER EXTRACTION COMPLETE")
print("=" * 90)

print(f"Total SMS processed : {len(results_df)}")
print(f"Successful JSON     : {successful}")
print(f"Failed extractions  : {failed}")
print(f"Success rate        : {(successful / len(results_df) * 100):.1f}%")
print(f"Total processing    : {total_time:.2f} seconds")

if len(results_df) > 0:
    print(
        f"Average latency    : "
        f"{results_df['latency_seconds'].mean():.3f} seconds/SMS"
    )

print(f"\nFinal CSV:")
print(OUTPUT_CSV)

# ------------------------------------------------------------
# 12. DISPLAY ANALYSIS-READY LEDGER
# ------------------------------------------------------------

print("\n")
print("=" * 90)
print("ANALYSIS-READY LEDGER")
print("=" * 90)

display(results_df)

# ------------------------------------------------------------
# 13. QUICK DATA QUALITY CHECK
# ------------------------------------------------------------

print("\n")
print("=" * 90)
print("DATA QUALITY CHECK")
print("=" * 90)

print("\nMissing values by field:")

missing_summary = (
    results_df[JSON_FIELDS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary.to_frame("missing_values")
)

print("\nTransaction types:")

display(
    results_df["type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nTransaction domains:")

display(
    results_df["domain"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nFinal output shape:", results_df.shape)

SME-LEDGER V2 — 50 SMS FINANCIAL LEDGER EXTRACTION

Input file: /kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv
Rows loaded: 50
Columns: ['message']
SMS column: 'message'
Samples to process: 50


STARTING SEQUENTIAL SMS EXTRACTION

------------------------------------------------------------------------------------------
SMS 1/50
------------------------------------------------------------------------------------------
SMS preview: TX03010001 Confirmed. You have received Ksh7,500.00 from John Kamau on 01/03/2026 at 08:18 AM. New M-PESA balance is Ksh157,500.00. Transaction cost, Ksh0.00.
Status       : ✗ EXTRACTION ERROR
Error        : Requested tokens (560) exceed context window of 512
Raw response : 
Latency      : 0.004s
Saved rows   : 1

------------------------------------------------------------------------------------------
SMS 2/50
------------------------------------------------------------------------------------------
SMS preview: TX03010002 Confir

,source_row,sample_number,sms,transaction_id,date,time,type,domain,entity,amount,balance,fee,reference,json_valid,extraction_error,latency_seconds
0,0,1,"TX03010001 Confirmed. You have received Ksh7,5...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (560) exceed context window o...,0.004
1,1,2,"TX03010002 Confirmed. You have received Ksh2,5...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (560) exceed context window o...,0.004
2,2,3,TX03020003 Confirmed. Ksh500.00 paid to Green ...,None,None,None,None,None,None,None,None,None,None,False,Requested tokens (556) exceed context window o...,0.004
3,3,4,TX03020004 Confirmed. Ksh500.00 sent to David ...,None,None,None,None,None,None,None,None,None,None,False,Requested tokens (558) exceed context window o...,0.004
4,4,5,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (558) exceed context window o...,0.003
5,5,6,"TX03030006 Confirmed. Ksh1,000.00 paid to Airt...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (558) exceed context window o...,0.004
6,6,7,"TX03040007 Confirmed. You have received Ksh2,5...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (560) exceed context window o...,0.004
7,7,8,TX03040008 Confirmed. Ksh500.00 sent to Peter ...,None,None,None,None,None,None,None,None,None,None,False,Requested tokens (557) exceed context window o...,0.004
8,8,9,"TX03050009 Confirmed. You have received Ksh7,5...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (560) exceed context window o...,0.004
9,9,10,"TX03050010 Confirmed. You have received Ksh2,5...",None,None,None,None,None,None,None,None,None,None,False,Requested tokens (560) exceed context window o...,0.004




DATA QUALITY CHECK

Missing values by field:


,missing_values
transaction_id,50
date,50
time,50
type,50
domain,50
entity,50
amount,50
balance,50
fee,50
reference,50



Transaction types:


,count
type,
None,50



Transaction domains:


,count
domain,
None,50



Final output shape: (50, 16)


## Financial analysis and dashboard
Run this cell after the `test.csv` inference cell. It builds a Pandas ledger, computes financial KPIs, displays charts and filters, and exports analysis CSVs.

In [6]:
# ============================================================
# SME-LEDGER V2 — FINANCIAL HEALTH ANALYTICS & DASHBOARD
#
# RUN AFTER CELL 5
#
# Cell 5 output:
#   /kaggle/working/sme_ledger_50_results.csv
#
# This cell:
#   1. Loads the structured model output
#   2. Cleans and validates the ledger
#   3. Computes financial-health KPIs
#   4. Analyzes income and expenditure
#   5. Analyzes liquidity / balance
#   6. Analyzes spending concentration
#   7. Analyzes transaction frequency
#   8. Analyzes fees
#   9. Flags financial-risk indicators
#  10. Produces interactive Plotly visualizations
#  11. Saves analysis-ready CSV files
#
# IMPORTANT:
#   This is descriptive financial analysis, not a lending decision
#   or financial advice system.
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_CSV = "/kaggle/working/sme_ledger_50_results.csv"

OUTPUT_LEDGER = "/kaggle/working/sme_ledger_final_analysis.csv"
OUTPUT_MONTHLY = "/kaggle/working/sme_ledger_monthly_analysis.csv"
OUTPUT_ENTITY = "/kaggle/working/sme_ledger_entity_analysis.csv"
OUTPUT_DOMAIN = "/kaggle/working/sme_ledger_domain_analysis.csv"
OUTPUT_HEALTH = "/kaggle/working/sme_ledger_financial_health.csv"

# ============================================================
# 2. LOAD STRUCTURED LEDGER
# ============================================================

try:
    ledger = pd.read_csv(INPUT_CSV)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find:\n{INPUT_CSV}\n\n"
        "Run Cell 5 first so the 50-SMS extraction CSV exists."
    )

print("=" * 90)
print("SME-LEDGER V2 — FINANCIAL HEALTH ANALYTICS")
print("=" * 90)

print(f"\nLoaded: {INPUT_CSV}")
print(f"Rows: {len(ledger):,}")
print(f"Columns: {len(ledger.columns)}")

# ============================================================
# 3. ENSURE EXPECTED COLUMNS EXIST
# ============================================================

expected_columns = [
    "sms",
    "transaction_id",
    "date",
    "time",
    "type",
    "domain",
    "entity",
    "amount",
    "balance",
    "fee",
    "reference",
    "json_valid",
    "extraction_error",
    "latency_seconds"
]

for col in expected_columns:

    if col not in ledger.columns:

        if col in ["amount", "balance", "fee", "latency_seconds"]:
            ledger[col] = np.nan

        elif col == "json_valid":
            ledger[col] = False

        else:
            ledger[col] = None

# ============================================================
# 4. CLEAN DATA TYPES
# ============================================================

def clean_numeric(series):

    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("Ksh", "", regex=False)
        .str.replace("KES", "", regex=False)
        .str.replace("ksh", "", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "none": np.nan,
            "null": np.nan
        }),
        errors="coerce"
    )


for col in ["amount", "balance", "fee"]:
    ledger[col] = clean_numeric(ledger[col])


# ============================================================
# 5. NORMALIZE TRANSACTION TYPE
# ============================================================

ledger["type"] = (
    ledger["type"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

ledger["direction"] = ledger["type"].map({

    "income": "Income",
    "received": "Income",
    "receive": "Income",
    "deposit": "Income",
    "credit": "Income",

    "expense": "Expense",
    "payment": "Expense",
    "paid": "Expense",
    "sent": "Expense",
    "withdrawal": "Expense",
    "debit": "Expense"

}).fillna("Unknown")


# ============================================================
# 6. CLEAN ENTITY / DOMAIN
# ============================================================

for col in ["entity", "domain"]:

    ledger[col] = (
        ledger[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown",
            "nan": "Unknown",
            "None": "Unknown",
            "null": "Unknown"
        })
    )


# ============================================================
# 7. DATE / TIME
# ============================================================

ledger["date_parsed"] = pd.to_datetime(
    ledger["date"],
    errors="coerce",
    dayfirst=True
)

# If dates were already YYYY-MM-DD, retry without dayfirst
missing_dates = ledger["date_parsed"].isna()

if missing_dates.any():

    ledger.loc[missing_dates, "date_parsed"] = pd.to_datetime(
        ledger.loc[missing_dates, "date"],
        errors="coerce"
    )

ledger["month"] = ledger["date_parsed"].dt.to_period("M").astype(str)

ledger.loc[
    ledger["date_parsed"].isna(),
    "month"
] = "Unknown date"

ledger["day"] = ledger["date_parsed"].dt.date


# ============================================================
# 8. CORE FINANCIAL VARIABLES
# ============================================================

ledger["income_kes"] = np.where(
    ledger["direction"].eq("Income"),
    ledger["amount"],
    0
)

ledger["expense_kes"] = np.where(
    ledger["direction"].eq("Expense"),
    ledger["amount"],
    0
)

ledger["net_cashflow_kes"] = (
    ledger["income_kes"] -
    ledger["expense_kes"]
)

ledger["fee_kes"] = ledger["fee"].fillna(0)

# ============================================================
# 9. TRANSACTION INDEX
# ============================================================

ledger["transaction_number"] = np.arange(
    1,
    len(ledger) + 1
)


# ============================================================
# 10. DATA QUALITY ANALYSIS
# ============================================================

ledger["missing_amount"] = ledger["amount"].isna()

ledger["missing_balance"] = ledger["balance"].isna()

ledger["missing_date"] = ledger["date_parsed"].isna()

ledger["unknown_direction"] = (
    ledger["direction"] == "Unknown"
)

ledger["invalid_json"] = (
    ledger["json_valid"].astype(str).str.lower()
    .isin(["false", "0", "nan", "none"])
)

# Duplicate references
ref_clean = (
    ledger["reference"]
    .fillna("")
    .astype(str)
    .str.strip()
)

valid_reference = (
    ref_clean.ne("") &
    ~ref_clean.str.lower().isin(
        ["nan", "none", "null"]
    )
)

ledger["possible_duplicate"] = False

ledger.loc[valid_reference, "possible_duplicate"] = (
    ref_clean[valid_reference]
    .duplicated(keep=False)
)

# ============================================================
# 11. BASIC FINANCIAL KPIs
# ============================================================

total_transactions = len(ledger)

valid_transactions = ledger["amount"].notna()

income_rows = ledger[
    ledger["direction"] == "Income"
]

expense_rows = ledger[
    ledger["direction"] == "Expense"
]

total_income = income_rows["amount"].sum()

total_expenses = expense_rows["amount"].sum()

net_cashflow = (
    total_income -
    total_expenses
)

average_transaction = (
    ledger.loc[
        valid_transactions,
        "amount"
    ].mean()
)

median_transaction = (
    ledger.loc[
        valid_transactions,
        "amount"
    ].median()
)

largest_income = (
    income_rows["amount"].max()
    if not income_rows.empty
    else np.nan
)

largest_expense = (
    expense_rows["amount"].max()
    if not expense_rows.empty
    else np.nan
)

income_count = len(income_rows)

expense_count = len(expense_rows)

income_expense_ratio = (
    total_income / total_expenses
    if total_expenses > 0
    else np.nan
)

# ============================================================
# 12. BALANCE / LIQUIDITY
# ============================================================

balance_rows = ledger[
    ledger["balance"].notna()
].copy()

if not balance_rows.empty:

    balance_rows = balance_rows.sort_values(
        ["date_parsed", "transaction_number"],
        na_position="last"
    )

    latest_balance = balance_rows["balance"].iloc[-1]

    lowest_balance = balance_rows["balance"].min()

    highest_balance = balance_rows["balance"].max()

    first_balance = balance_rows["balance"].iloc[0]

    balance_change = (
        latest_balance -
        first_balance
    )

else:

    latest_balance = np.nan
    lowest_balance = np.nan
    highest_balance = np.nan
    first_balance = np.nan
    balance_change = np.nan


# ============================================================
# 13. CASH-FLOW MARGIN
# ============================================================

cashflow_margin = (
    net_cashflow / total_income * 100
    if total_income > 0
    else np.nan
)


# ============================================================
# 14. EXPENSE CONCENTRATION
# ============================================================

expense_entities = (
    expense_rows
    .groupby("entity", dropna=False)["amount"]
    .sum()
    .sort_values(ascending=False)
)

if len(expense_entities) > 0:

    largest_expense_entity = expense_entities.index[0]

    largest_entity_spend = expense_entities.iloc[0]

    largest_entity_share = (
        largest_entity_spend /
        total_expenses * 100
        if total_expenses > 0
        else np.nan
    )

else:

    largest_expense_entity = "N/A"
    largest_entity_spend = np.nan
    largest_entity_share = np.nan


# ============================================================
# 15. DOMAIN ANALYSIS
# ============================================================

domain_expenses = (
    expense_rows
    .groupby("domain", dropna=False)["amount"]
    .agg(
        total_spend="sum",
        transaction_count="count",
        average_transaction="mean"
    )
    .sort_values(
        "total_spend",
        ascending=False
    )
    .reset_index()
)

domain_income = (
    income_rows
    .groupby("domain", dropna=False)["amount"]
    .agg(
        total_income="sum",
        transaction_count="count",
        average_transaction="mean"
    )
    .sort_values(
        "total_income",
        ascending=False
    )
    .reset_index()
)


# ============================================================
# 16. MONTHLY CASH-FLOW ANALYSIS
# ============================================================

monthly = (
    ledger[
        ledger["month"] != "Unknown date"
    ]
    .groupby("month")
    .agg(
        income_kes=("income_kes", "sum"),
        expense_kes=("expense_kes", "sum"),
        net_cashflow_kes=("net_cashflow_kes", "sum"),
        transaction_count=("amount", "count"),
        total_fees_kes=("fee_kes", "sum")
    )
    .reset_index()
)

if not monthly.empty:

    monthly["cashflow_margin_pct"] = np.where(
        monthly["income_kes"] > 0,
        (
            monthly["net_cashflow_kes"] /
            monthly["income_kes"]
        ) * 100,
        np.nan
    )

    monthly["income_expense_ratio"] = np.where(
        monthly["expense_kes"] > 0,
        monthly["income_kes"] /
        monthly["expense_kes"],
        np.nan
    )


# ============================================================
# 17. FINANCIAL HEALTH INDICATORS
# ============================================================

# These are descriptive indicators rather than credit scores.

positive_cashflow = (
    net_cashflow > 0
)

expense_to_income_pct = (
    total_expenses /
    total_income * 100
    if total_income > 0
    else np.nan
)

fee_to_transaction_pct = (
    ledger["fee_kes"].sum() /
    total_income * 100
    if total_income > 0
    else np.nan
)

# Balance stability
if not balance_rows.empty:

    balance_std = balance_rows["balance"].std()

    balance_mean = balance_rows["balance"].mean()

    balance_volatility_pct = (
        balance_std /
        balance_mean * 100
        if balance_mean and balance_mean > 0
        else np.nan
    )

else:

    balance_std = np.nan
    balance_mean = np.nan
    balance_volatility_pct = np.nan


# ============================================================
# 18. FINANCIAL HEALTH SUMMARY
# ============================================================

health_metrics = pd.DataFrame({

    "Metric": [

        "Total transactions",
        "Income transactions",
        "Expense transactions",

        "Total income (KES)",
        "Total expenses (KES)",
        "Net cash flow (KES)",

        "Cash-flow margin (%)",
        "Expense / income (%)",
        "Income / expense ratio",

        "Average transaction (KES)",
        "Median transaction (KES)",

        "Largest income (KES)",
        "Largest expense (KES)",

        "First extracted balance (KES)",
        "Latest extracted balance (KES)",
        "Lowest extracted balance (KES)",
        "Highest extracted balance (KES)",
        "Balance change (KES)",

        "Total fees (KES)",
        "Fees / income (%)",

        "Largest expense entity",
        "Largest entity share of spending (%)",

        "Transactions with missing amount",
        "Transactions with missing date",
        "Unknown transaction direction",
        "Possible duplicate references",
        "Invalid JSON extractions"

    ],

    "Value": [

        total_transactions,
        income_count,
        expense_count,

        total_income,
        total_expenses,
        net_cashflow,

        cashflow_margin,
        expense_to_income_pct,
        income_expense_ratio,

        average_transaction,
        median_transaction,

        largest_income,
        largest_expense,

        first_balance,
        latest_balance,
        lowest_balance,
        highest_balance,
        balance_change,

        ledger["fee_kes"].sum(),
        fee_to_transaction_pct,

        largest_expense_entity,
        largest_entity_share,

        int(ledger["missing_amount"].sum()),
        int(ledger["missing_date"].sum()),
        int(ledger["unknown_direction"].sum()),
        int(ledger["possible_duplicate"].sum()),
        int(ledger["invalid_json"].sum())

    ]

})

# ============================================================
# 19. DASHBOARD HEADER
# ============================================================

display(
    Markdown(
        "# 💰 SME-Ledger V2 — Financial Health Dashboard"
    )
)

display(
    Markdown(
        f"""
### Dataset overview

**Transactions:** {total_transactions:,}

**Income:** KES {total_income:,.2f}

**Expenses:** KES {total_expenses:,.2f}

**Net cash flow:** KES {net_cashflow:,.2f}

**Latest extracted balance:** 
KES {latest_balance:,.2f}
"""
        if pd.notna(latest_balance)
        else
        f"""
### Dataset overview

**Transactions:** {total_transactions:,}

**Income:** KES {total_income:,.2f}

**Expenses:** KES {total_expenses:,.2f}

**Net cash flow:** KES {net_cashflow:,.2f}

**Latest extracted balance:** N/A
"""
    )
)

# ============================================================
# 20. KPI TABLE
# ============================================================

display(
    Markdown("## 📊 Core Financial KPIs")
)

kpi_display = pd.DataFrame({

    "KPI": [
        "Total income",
        "Total expenses",
        "Net cash flow",
        "Average transaction",
        "Median transaction",
        "Income / expense ratio",
        "Cash-flow margin",
        "Latest balance"
    ],

    "Value": [

        f"KES {total_income:,.2f}",

        f"KES {total_expenses:,.2f}",

        f"KES {net_cashflow:,.2f}",

        (
            f"KES {average_transaction:,.2f}"
            if pd.notna(average_transaction)
            else "N/A"
        ),

        (
            f"KES {median_transaction:,.2f}"
            if pd.notna(median_transaction)
            else "N/A"
        ),

        (
            f"{income_expense_ratio:.2f}"
            if pd.notna(income_expense_ratio)
            else "N/A"
        ),

        (
            f"{cashflow_margin:.2f}%"
            if pd.notna(cashflow_margin)
            else "N/A"
        ),

        (
            f"KES {latest_balance:,.2f}"
            if pd.notna(latest_balance)
            else "N/A"
        )

    ]
})

display(kpi_display)


# ============================================================
# 21. FINANCIAL HEALTH OBSERVATIONS
# ============================================================

display(
    Markdown("## 🔎 Financial Health Indicators")
)

observations = []

if positive_cashflow:
    observations.append(
        "🟢 **Positive net cash flow:** "
        "total extracted income exceeds total extracted expenses."
    )
elif net_cashflow < 0:
    observations.append(
        "🔴 **Negative net cash flow:** "
        "extracted expenses exceed extracted income."
    )
else:
    observations.append(
        "🟡 **Neutral cash flow:** "
        "extracted income and expenses are approximately balanced."
    )

if pd.notna(expense_to_income_pct):

    if expense_to_income_pct > 100:
        observations.append(
            "🔴 **Expenses exceed extracted income.**"
        )

    elif expense_to_income_pct > 80:
        observations.append(
            "🟠 **High expense-to-income ratio:** "
            f"expenses represent approximately "
            f"{expense_to_income_pct:.1f}% of extracted income."
        )

    else:
        observations.append(
            "🟢 **Expense-to-income ratio:** "
            f"approximately {expense_to_income_pct:.1f}%."
        )

if pd.notna(balance_change):

    if balance_change > 0:
        observations.append(
            f"🟢 **Balance increased:** "
            f"approximately KES {balance_change:,.2f} "
            "between the first and latest extracted balances."
        )

    elif balance_change < 0:
        observations.append(
            f"🟠 **Balance decreased:** "
            f"approximately KES {abs(balance_change):,.2f}."
        )

if pd.notna(largest_entity_share):

    if largest_entity_share > 50:
        observations.append(
            f"🟠 **Spending concentration:** "
            f"{largest_expense_entity} accounts for approximately "
            f"{largest_entity_share:.1f}% of extracted spending."
        )

if ledger["missing_amount"].sum() > 0:

    observations.append(
        f"🟡 **Data completeness:** "
        f"{int(ledger['missing_amount'].sum())} transaction(s) "
        "have no extracted amount."
    )

if ledger["invalid_json"].sum() > 0:

    observations.append(
        f"🟡 **Extraction quality:** "
        f"{int(ledger['invalid_json'].sum())} row(s) "
        "were not successfully parsed as JSON."
    )

for item in observations:
    display(Markdown(f"- {item}"))


# ============================================================
# 22. CHART 1 — INCOME VS EXPENSE
# ============================================================

if not monthly.empty:

    monthly_long = monthly.melt(
        id_vars=["month"],
        value_vars=[
            "income_kes",
            "expense_kes"
        ],
        var_name="flow",
        value_name="KES"
    )

    monthly_long["flow"] = monthly_long["flow"].map({
        "income_kes": "Income",
        "expense_kes": "Expenses"
    })

    fig = px.bar(
        monthly_long,
        x="month",
        y="KES",
        color="flow",
        barmode="group",
        title="Monthly Income vs Expenses"
    )

    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="KES",
        hovermode="x unified"
    )

    fig.show()


# ============================================================
# 23. CHART 2 — NET CASH FLOW
# ============================================================

if not monthly.empty:

    fig = px.bar(
        monthly,
        x="month",
        y="net_cashflow_kes",
        title="Monthly Net Cash Flow",
        labels={
            "net_cashflow_kes": "Net Cash Flow (KES)",
            "month": "Month"
        }
    )

    fig.add_hline(
        y=0,
        line_dash="dash"
    )

    fig.show()


# ============================================================
# 24. CHART 3 — BALANCE TREND
# ============================================================

balances = (
    ledger[
        ledger["balance"].notna()
    ]
    .sort_values(
        ["date_parsed", "transaction_number"],
        na_position="last"
    )
)

if not balances.empty:

    fig = px.line(
        balances,
        x="date_parsed",
        y="balance",
        markers=True,
        title="Extracted Account Balance Over Time",
        labels={
            "date_parsed": "Date",
            "balance": "Balance (KES)"
        }
    )

    fig.show()


# ============================================================
# 25. CHART 4 — INCOME / EXPENSE TRANSACTION COUNTS
# ============================================================

direction_counts = (
    ledger["direction"]
    .value_counts()
    .rename_axis("Direction")
    .reset_index(name="Transactions")
)

fig = px.bar(
    direction_counts,
    x="Direction",
    y="Transactions",
    title="Transaction Frequency by Direction"
)

fig.show()


# ============================================================
# 26. CHART 5 — EXPENSES BY DOMAIN
# ============================================================

spending_domain = (
    expense_rows
    .groupby("domain")["amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

if not spending_domain.empty:

    fig = px.pie(
        spending_domain,
        names="domain",
        values="amount",
        hole=0.45,
        title="Where Is the Money Being Spent?"
    )

    fig.show()


# ============================================================
# 27. CHART 6 — TOP EXPENSE ENTITIES
# ============================================================

top_entities = (
    expense_rows
    .groupby("entity")["amount"]
    .sum()
    .nlargest(10)
    .sort_values()
    .reset_index()
)

if not top_entities.empty:

    fig = px.bar(
        top_entities,
        x="amount",
        y="entity",
        orientation="h",
        title="Top 10 Expense Entities",
        labels={
            "amount": "Total Spending (KES)",
            "entity": "Entity"
        }
    )

    fig.show()


# ============================================================
# 28. CHART 7 — TRANSACTION SIZE DISTRIBUTION
# ============================================================

amount_data = ledger[
    ledger["amount"].notna()
].copy()

if not amount_data.empty:

    fig = px.histogram(
        amount_data,
        x="amount",
        color="direction",
        nbins=20,
        title="Transaction Amount Distribution",
        labels={
            "amount": "Transaction Amount (KES)"
        }
    )

    fig.show()


# ============================================================
# 29. CHART 8 — DAILY CASH FLOW
# ============================================================

daily = (
    ledger[
        ledger["date_parsed"].notna()
    ]
    .groupby("day")
    .agg(
        income_kes=("income_kes", "sum"),
        expense_kes=("expense_kes", "sum"),
        net_cashflow_kes=("net_cashflow_kes", "sum")
    )
    .reset_index()
)

if not daily.empty:

    daily_long = daily.melt(
        id_vars="day",
        value_vars=[
            "income_kes",
            "expense_kes"
        ],
        var_name="flow",
        value_name="KES"
    )

    daily_long["flow"] = daily_long["flow"].map({
        "income_kes": "Income",
        "expense_kes": "Expenses"
    })

    fig = px.bar(
        daily_long,
        x="day",
        y="KES",
        color="flow",
        barmode="group",
        title="Daily Income and Expenses"
    )

    fig.show()


# ============================================================
# 30. CHART 9 — FEES
# ============================================================

fees_by_domain = (
    ledger
    .groupby("domain")["fee_kes"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fees_by_domain = fees_by_domain[
    fees_by_domain["fee_kes"] > 0
]

if not fees_by_domain.empty:

    fig = px.bar(
        fees_by_domain,
        x="domain",
        y="fee_kes",
        title="Transaction Fees by Financial Domain",
        labels={
            "fee_kes": "Fees (KES)",
            "domain": "Domain"
        }
    )

    fig.show()


# ============================================================
# 31. CHART 10 — CASH FLOW MARGIN
# ============================================================

if not monthly.empty:

    fig = px.line(
        monthly,
        x="month",
        y="cashflow_margin_pct",
        markers=True,
        title="Monthly Cash-Flow Margin",
        labels={
            "cashflow_margin_pct": "Cash-Flow Margin (%)",
            "month": "Month"
        }
    )

    fig.add_hline(
        y=0,
        line_dash="dash"
    )

    fig.show()


# ============================================================
# 32. TOP INCOME SOURCES
# ============================================================

top_income_entities = (
    income_rows
    .groupby("entity")["amount"]
    .sum()
    .nlargest(10)
    .sort_values()
    .reset_index()
)

if not top_income_entities.empty:

    fig = px.bar(
        top_income_entities,
        x="amount",
        y="entity",
        orientation="h",
        title="Top 10 Income Sources",
        labels={
            "amount": "Income (KES)",
            "entity": "Entity"
        }
    )

    fig.show()


# ============================================================
# 33. ENTITY-LEVEL ANALYSIS
# ============================================================

entity_analysis = (
    ledger
    .groupby(
        ["entity", "direction"],
        dropna=False
    )
    .agg(
        total_amount=("amount", "sum"),
        transaction_count=("amount", "count"),
        average_amount=("amount", "mean"),
        total_fees=("fee_kes", "sum")
    )
    .reset_index()
)

entity_analysis.to_csv(
    OUTPUT_ENTITY,
    index=False
)


# ============================================================
# 34. DOMAIN-LEVEL ANALYSIS
# ============================================================

domain_analysis = (
    ledger
    .groupby(
        ["domain", "direction"],
        dropna=False
    )
    .agg(
        total_amount=("amount", "sum"),
        transaction_count=("amount", "count"),
        average_amount=("amount", "mean"),
        total_fees=("fee_kes", "sum")
    )
    .reset_index()
)

domain_analysis.to_csv(
    OUTPUT_DOMAIN,
    index=False
)


# ============================================================
# 35. FINANCIAL HEALTH DATASET
# ============================================================

health_record = {

    "total_transactions":
        total_transactions,

    "income_transactions":
        income_count,

    "expense_transactions":
        expense_count,

    "total_income_kes":
        total_income,

    "total_expenses_kes":
        total_expenses,

    "net_cashflow_kes":
        net_cashflow,

    "cashflow_margin_pct":
        cashflow_margin,

    "expense_to_income_pct":
        expense_to_income_pct,

    "income_expense_ratio":
        income_expense_ratio,

    "average_transaction_kes":
        average_transaction,

    "median_transaction_kes":
        median_transaction,

    "largest_income_kes":
        largest_income,

    "largest_expense_kes":
        largest_expense,

    "first_balance_kes":
        first_balance,

    "latest_balance_kes":
        latest_balance,

    "lowest_balance_kes":
        lowest_balance,

    "highest_balance_kes":
        highest_balance,

    "balance_change_kes":
        balance_change,

    "balance_volatility_pct":
        balance_volatility_pct,

    "total_fees_kes":
        ledger["fee_kes"].sum(),

    "fee_to_income_pct":
        fee_to_transaction_pct,

    "largest_expense_entity":
        largest_expense_entity,

    "largest_entity_spend_kes":
        largest_entity_spend,

    "largest_entity_share_pct":
        largest_entity_share,

    "missing_amount_count":
        int(ledger["missing_amount"].sum()),

    "missing_date_count":
        int(ledger["missing_date"].sum()),

    "unknown_direction_count":
        int(ledger["unknown_direction"].sum()),

    "possible_duplicate_count":
        int(ledger["possible_duplicate"].sum()),

    "invalid_json_count":
        int(ledger["invalid_json"].sum())

}

financial_health = pd.DataFrame(
    [health_record]
)

financial_health.to_csv(
    OUTPUT_HEALTH,
    index=False
)


# ============================================================
# 36. SAVE MONTHLY ANALYSIS
# ============================================================

monthly.to_csv(
    OUTPUT_MONTHLY,
    index=False
)


# ============================================================
# 37. SAVE FINAL ANALYSIS-READY LEDGER
# ============================================================

ledger.to_csv(
    OUTPUT_LEDGER,
    index=False
)


# ============================================================
# 38. TRANSACTION TABLE
# ============================================================

display(
    Markdown(
        "## 📋 Analysis-Ready Transaction Ledger"
    )
)

display(
    ledger[
        [
            "transaction_number",
            "sms",
            "date",
            "time",
            "type",
            "domain",
            "entity",
            "amount",
            "balance",
            "fee",
            "reference",
            "json_valid"
        ]
    ].head(100)
)


# ============================================================
# 39. DATA QUALITY REPORT
# ============================================================

display(
    Markdown(
        "## 🧪 Extraction & Data Quality"
    )
)

quality = pd.DataFrame({

    "Check": [

        "Total rows",
        "Valid JSON",
        "Invalid JSON",
        "Missing amount",
        "Missing balance",
        "Missing date",
        "Unknown direction",
        "Possible duplicate reference"

    ],

    "Rows": [

        len(ledger),

        int(ledger["json_valid"].sum()),

        int(ledger["invalid_json"].sum()),

        int(ledger["missing_amount"].sum()),

        int(ledger["missing_balance"].sum()),

        int(ledger["missing_date"].sum()),

        int(ledger["unknown_direction"].sum()),

        int(ledger["possible_duplicate"].sum())

    ]

})

display(quality)

# ============================================================
# 40. OUTPUT FILES
# ============================================================

print("\n")
print("=" * 90)
print("ANALYSIS COMPLETE")
print("=" * 90)

print(f"\nFinal transaction ledger:")
print(OUTPUT_LEDGER)

print("\nMonthly analysis:")
print(OUTPUT_MONTHLY)

print("\nEntity analysis:")
print(OUTPUT_ENTITY)

print("\nDomain analysis:")
print(OUTPUT_DOMAIN)

print("\nFinancial health summary:")
print(OUTPUT_HEALTH)

print("\nFinal ledger shape:", ledger.shape)

SME-LEDGER V2 — FINANCIAL HEALTH ANALYTICS

Loaded: /kaggle/working/sme_ledger_50_results.csv
Rows: 50
Columns: 16


/tmp/ipykernel_16/2073597281.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_16/2073597281.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
/tmp/ipykernel_16/2073597281.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({


# 💰 SME-Ledger V2 — Financial Health Dashboard


### Dataset overview

**Transactions:** 50

**Income:** KES 0.00

**Expenses:** KES 0.00

**Net cash flow:** KES 0.00

**Latest extracted balance:** N/A


## 📊 Core Financial KPIs

...

## 📋 Analysis-Ready Transaction Ledger

,transaction_number,sms,date,time,type,domain,entity,amount,balance,fee,reference,json_valid
0,1,"TX03010001 Confirmed. You have received Ksh7,5...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
1,2,"TX03010002 Confirmed. You have received Ksh2,5...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
2,3,TX03020003 Confirmed. Ksh500.00 paid to Green ...,NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
3,4,TX03020004 Confirmed. Ksh500.00 sent to David ...,NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
4,5,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
5,6,"TX03030006 Confirmed. Ksh1,000.00 paid to Airt...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
6,7,"TX03040007 Confirmed. You have received Ksh2,5...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
7,8,TX03040008 Confirmed. Ksh500.00 sent to Peter ...,NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
8,9,"TX03050009 Confirmed. You have received Ksh7,5...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False
9,10,"TX03050010 Confirmed. You have received Ksh2,5...",NaN,NaN,unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,False


## 🧪 Extraction & Data Quality

,Check,Rows
0,Total rows,50
1,Valid JSON,0
2,Invalid JSON,50
3,Missing amount,50
4,Missing balance,50
5,Missing date,50
6,Unknown direction,50
7,Possible duplicate reference,0




ANALYSIS COMPLETE

Final transaction ledger:
/kaggle/working/sme_ledger_final_analysis.csv

Monthly analysis:
/kaggle/working/sme_ledger_monthly_analysis.csv

Entity analysis:
/kaggle/working/sme_ledger_entity_analysis.csv

Domain analysis:
/kaggle/working/sme_ledger_domain_analysis.csv

Financial health summary:
/kaggle/working/sme_ledger_financial_health.csv

Final ledger shape: (50, 31)
